### IMPORTS

In [1]:
# IMPORTS
from dataset_manager import Dataset_Manager
import io
import numpy as np
import os
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix
from sklearn.neural_network import MLPClassifier
import subprocess
import sys

### DATASETS

In [2]:
# LOAD DATASETS
dataset_manager = Dataset_Manager(reload_recola=False, reload_again=False)
RECOLA_DATASET = dataset_manager.recola_dataset
AGAIN_DATASET = dataset_manager.again_dataset

### INVARIANT FEATURES

In [ ]:
# GET INVARIANT FEATURES
def get_invariant_features(input_file: str, output_file: str):
    # STEP 1: SET UP R SCRIPT CONSTANTS
    RSCRIPT = "C:/Program Files/R/R-4.4.3/bin/x64/Rscript.exe"
    INVARIANT_FEATURES_FILE = "invariant_features.R" 
    run_params = [RSCRIPT, INVARIANT_FEATURES_FILE, "inv_pred_recola", input_file, output_file]

    # STEP 2: RUN THE R SCRIPT
    try:
        subprocess.run(run_params, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e: 
        print("Error running R script:", e)

    # STEP 3: OBTAIN INVARIANT FEATURES AND CLEAN TEMPORARY FILES
    invariant_features_RECOLA = pd.read_csv(output_file)
    os.remove(input_file)
    os.remove(output_file)

    return list(invariant_features_RECOLA["invariant_feature_names"])

### MODEL TRAINING BLOCKS

In [ ]:
def logistic_regression_run(invariant_flag: bool, dataset: pd.DataFrame, groups: list, features: pd.DataFrame, target: pd.Series, continuous_target: pd.Series, folds: int, display_inv_flag: bool):
    conf_matrix_list = list()
    pcc_list = list()
    
    # STEP 1: SETTING UP CROSS FOLD VALIDATION
    group_k_fold = GroupKFold(n_splits=folds)
    fold_counter = 1
    for train_index, test_index in group_k_fold.split(features, target, groups):

        if invariant_flag is True:
            fold_dataframe = dataset.iloc[train_index]
            fold_dataframe.to_csv("temporary_dataset.csv")
            invariant_features = get_invariant_features(input_file="temporary_dataset.csv", output_file="temporary_invariant_features.csv")
            
            if display_inv_flag is True:
                print(f"INVARIANT FEATURES OF (FOLD {fold_counter}) - NUMBER OF FEATURES: {len(invariant_features)}")
                print(f"{invariant_features}\n")
                print()

            fold_features = features[invariant_features]
            
        elif invariant_flag is False:
            fold_features = features

        if not fold_features.empty:
            # STEP 2: OBTAINING TRAINING DATA FOR FOLD
            feature_train = fold_features.iloc[train_index]
            target_train = target.iloc[train_index]

            # STEP 3: OBTAINING TESTING DATA FOR FOLD
            feature_test = fold_features.iloc[test_index]
            target_test = target.iloc[test_index]
            contin_target_test = continuous_target.iloc[test_index]

            model = LogisticRegression(solver='liblinear', max_iter=100000)
            model.fit(X=feature_train, y=target_train)

            test_prediction = model.predict(feature_test)

            test_porb_prediction = model.predict_proba(feature_test)[:, 1]
            pcc, _ = stats.pearsonr(test_porb_prediction, contin_target_test)
            pcc_list.append(pcc)

            conf_matrix = confusion_matrix(target_test, test_prediction)
            conf_matrix_list.append(conf_matrix) 

        fold_counter += 1

    # Averaging confusion matrices
    if len(conf_matrix_list)  > 0:
        mean_confusion_matrix = np.mean(conf_matrix_list, axis=0)
        true_neg, false_pos, false_neg, true_pos = mean_confusion_matrix.ravel()

        mean_accuracy = (true_pos + true_neg) / np.sum(mean_confusion_matrix)
        mean_precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) else 0
        mean_recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) else 0

        mean_pcc = np.mean(pcc_list)

        return mean_accuracy, mean_precision, mean_recall, mean_pcc
    else:
        return None, None, None, None

def run_logistic_regression_tests(dataset: pd.DataFrame, groups: list, features: pd.DataFrame, target: pd.Series, continuous_target: pd.Series, folds: int, display_inv_flag: bool):
    print("-------- STANDARD LOGISTIC REGRESSION --------")
    accuracy, precision, recall, pcc = logistic_regression_run(invariant_flag=False, dataset=dataset, groups=groups, features=features, target=target, continuous_target=continuous_target, folds=folds, display_inv_flag=display_inv_flag)
    print(f" AVERAGE ACCURACY: {accuracy}\n AVERAGE PRECISION: {precision}\n AVERAGE RECALL: {recall}\n AVERAGE PCC: {pcc}\n")
    
    print("-------- INVARIANT LOGISTIC REGRESSION --------")
    invariant_accuracy, invariant_precision, invariant_recall, invariant_pcc = logistic_regression_run(invariant_flag=True, dataset=dataset, groups=groups, features=features, target=target, continuous_target=continuous_target, folds=folds, display_inv_flag=display_inv_flag)
    print(f" AVERAGE ACCURACY: {invariant_accuracy}\n AVERAGE PRECISION: {invariant_precision}\n AVERAGE RECALL: {invariant_recall}\n AVERAGE PCC: {invariant_pcc}\n")

    print("-------- COMPARING STANDARD AND INVARIANT RESULTS --------")
    print(f" ACCURACY:\t{accuracy} -> {invariant_accuracy}")
    print(f" PRECISION:\t{precision} -> {invariant_precision}")
    print(f" RECALL:\t{recall} -> {invariant_recall}")
    print(f" PCC:\t{pcc} -> {invariant_pcc}")

In [ ]:
def neural_networks_run(invariant_flag: bool, dataset: pd.DataFrame, groups: list, features: pd.DataFrame, target: pd.Series, continuous_target: pd.Series, folds: int, display_inv_flag: bool):
    conf_matrix_list = list()
    pcc_list = list()
    
    # STEP 1: SETTING UP CROSS FOLD VALIDATION
    group_k_fold = GroupKFold(n_splits=folds)
    fold_counter = 1
    for train_index, test_index in group_k_fold.split(features, target, groups):

        if invariant_flag is True:
            fold_dataframe = dataset.iloc[train_index]
            fold_dataframe.to_csv("temporary_dataset.csv")
            invariant_features = get_invariant_features(input_file="temporary_dataset.csv", output_file="temporary_invariant_features.csv")
            
            if display_inv_flag is True:
                print(f"INVARIANT FEATURES OF (FOLD {fold_counter}) - NUMBER OF FEATURES: {len(invariant_features)}")
                print(f"{invariant_features}\n")
                
            fold_features = features[invariant_features]
            
        elif invariant_flag is False:
            fold_features = features

        if not fold_features.empty:

            # STEP 2: OBTAINING TRAINING DATA FOR FOLD
            feature_train = fold_features.iloc[train_index]
            target_train = target.iloc[train_index]

            # STEP 3: OBTAINING TESTING DATA FOR FOLD
            feature_test = fold_features.iloc[test_index]
            target_test = target.iloc[test_index]
            contin_target_test = continuous_target.iloc[test_index]

            model = MLPClassifier(hidden_layer_sizes=(32,), max_iter=10000, random_state=42)
            model.fit(X=feature_train, y=target_train)

            test_prediction = model.predict(feature_test)
            
            test_porb_prediction = model.predict_proba(feature_test)[:, 1]
            pcc, _ = stats.pearsonr(test_porb_prediction, contin_target_test)
            pcc_list.append(pcc)

            conf_matrix = confusion_matrix(target_test, test_prediction)
            conf_matrix_list.append(conf_matrix) 

        fold_counter += 1

    # Averaging confusion matrices
    if len(conf_matrix_list)  > 0:
        mean_confusion_matrix = np.mean(conf_matrix_list, axis=0)
        true_neg, false_pos, false_neg, true_pos = mean_confusion_matrix.ravel()

        mean_accuracy = (true_pos + true_neg) / np.sum(mean_confusion_matrix)
        mean_precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) else 0
        mean_recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) else 0

        mean_pcc = np.mean(pcc_list)

        return mean_accuracy, mean_precision, mean_recall, mean_pcc
    else:
        return None, None, None, None


def run_neural_networks_tests(dataset: pd.DataFrame, groups: list, features: pd.DataFrame, target: pd.Series, continuous_target: pd.Series, folds: int, display_inv_flag: bool):
    print("-------- STANDARD NEURAL NETWORKS --------")
    accuracy, precision, recall, pcc = neural_networks_run(invariant_flag=False, dataset=dataset, groups=groups, features=features, target=target, continuous_target=continuous_target, folds=folds, display_inv_flag=display_inv_flag)
    print(f" AVERAGE ACCURACY: {accuracy}\n AVERAGE PRECISION: {precision}\n AVERAGE RECALL: {recall}\n AVERAGE PCC: {pcc}\n")
    
    print("-------- INVARIANT NEURAL NETWORKS --------")
    invariant_accuracy, invariant_precision, invariant_recall, invariant_pcc = neural_networks_run(invariant_flag=True, dataset=dataset, groups=groups, features=features, target=target, continuous_target=continuous_target, folds=folds, display_inv_flag=display_inv_flag)
    print(f" AVERAGE ACCURACY: {invariant_accuracy}\n AVERAGE PRECISION: {invariant_precision}\n AVERAGE RECALL: {invariant_recall}\n AVERAGE PCC: {invariant_pcc}\n")

    print("-------- COMPARING STANDARD AND INVARIANT RESULTS --------")
    print(f" ACCURACY:\t{accuracy} -> {invariant_accuracy}")
    print(f" PRECISION:\t{precision} -> {invariant_precision}")
    print(f" RECALL:\t{recall} -> {invariant_recall}")
    print(f" PCC:\t{pcc} -> {invariant_pcc}")

### RECOLA EXPERIMENT

In [ ]:
# TEST CASE 1: SINGLE MODALITY (AUDIO), 6 PARTICIPANTS, VALENCE
test_name = "RECOLA_Test_Case_1"

# STEP 1:  SETUP UP FILE SAVING FOR RESULTS
buffer = io.StringIO()
sys.stdout = buffer
print("-------- TEST CASE 1: SINGLE MODALITY (AUDIO), 6 PARTICIPANTS, TARGET LABEL: VALENCE --------")

# STEP 2: OBTAIN CASE DATASET
dataset_manager.temproary_recola = dataset_manager.get_number_of_participants_frame(number_of_participants=6)
dataset_manager.temproary_recola = dataset_manager.remove_modality(modalities=["Video", "Physiology"])
dataset_manager.temproary_recola = dataset_manager.remove_class_label(label_to_keep="Valence")
dataset = dataset_manager.save_custom_dataframe(file_name=f"Test_Datasets/{test_name}.csv")

# STEP 3: DATA PREP FOR MODELS
PARTICIPANT_GROUPS = list(dataset["Participant_Number"])
FEATURES = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
if "Class_Label_Valence" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Valence"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Valence"]
elif "Class_Label_Arousal" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Arousal"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Arousal"]
FOLDS = len(set(PARTICIPANT_GROUPS))

# STEP 4: DECIDE ON OUTPUT OF THE EXPERIMENTS
PRINT_INV_FEATURES  = False # True to display them, false not to

# STEP 5: RUN LOGISITIC REGRESSION MODEL
print("\nLOGISTIC REGRESSION TESTS")
run_logistic_regression_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 6: RUN NEURAL NETWORK MODEL
print("\nNEURAL NETWORKS TESTS")
run_neural_networks_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 7: SAVE OUTPUT
sys.stdout = sys.__stdout__
with open(f"Test_Results/{test_name}.txt", "w") as f:
    f.write(buffer.getvalue())

In [ ]:
# TEST CASE 2: MALE PARTICIPANTS, TARGET LABEL: AROUSAL
test_name = "RECOLA_Test_Case_2"

# STEP 1:  SETUP UP FILE SAVING FOR RESULTS
buffer = io.StringIO()
sys.stdout = buffer
print("-------- TEST CASE 2: MALE PARTICIPANTS, TARGET LABEL: AROUSAL --------\n")

# STEP 2: OBTAIN CASE DATASET
dataset_manager.temproary_recola = dataset_manager.split_by_gender(gender="Male")
dataset_manager.temproary_recola = dataset_manager.remove_class_label(label_to_keep="Arousal")
dataset = dataset_manager.save_custom_dataframe(file_name=f"{test_name}.csv")

# STEP 3: DATA PREP FOR MODELS
PARTICIPANT_GROUPS = list(dataset["Participant_Number"])
FEATURES = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
if "Class_Label_Valence" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Valence"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Valence"]
elif "Class_Label_Arousal" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Arousal"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Arousal"]
FOLDS = len(set(PARTICIPANT_GROUPS))

# STEP 4: DECIDE ON OUTPUT OF THE EXPERIMENTS
PRINT_INV_FEATURES  = False # True to display them, false not to

# STEP 5: RUN LOGISITIC REGRESSION MODEL
print("\nLOGISTIC REGRESSION TESTS")
run_logistic_regression_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 6: RUN NEURAL NETWORK MODEL
print("\nNEURAL NETWORKS TESTS")
run_neural_networks_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 7: SAVE OUTPUT
sys.stdout = sys.__stdout__
with open(f"Test_Results/{test_name}.txt", "w") as f:
    f.write(buffer.getvalue())

In [ ]:
# TEST CASE 3: MALE PARTICIPANTS, TARGET LABEL: VALENCE
test_name = "RECOLA_Test_Case_3"

# STEP 1:  SETUP UP FILE SAVING FOR RESULTS
buffer = io.StringIO()
sys.stdout = buffer
print("-------- TEST CASE 3: MALE PARTICIPANTS, TARGET LABEL: VALENCE --------\n")

# STEP 2: OBTAIN CASE DATASET
dataset_manager.temproary_recola = dataset_manager.split_by_gender(gender="Male")
dataset_manager.temproary_recola = dataset_manager.remove_class_label(label_to_keep="Valence")
dataset = dataset_manager.save_custom_dataframe(file_name=f"{test_name}.csv")

# STEP 3: DATA PREP FOR MODELS
PARTICIPANT_GROUPS = list(dataset["Participant_Number"])
FEATURES = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
if "Class_Label_Valence" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Valence"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Valence"]
elif "Class_Label_Arousal" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Arousal"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Arousal"]
FOLDS = len(set(PARTICIPANT_GROUPS))

# STEP 4: DECIDE ON OUTPUT OF THE EXPERIMENTS
PRINT_INV_FEATURES  = False # True to display them, false not to

# STEP 5: RUN LOGISITIC REGRESSION MODEL
print("\nLOGISTIC REGRESSION TESTS")
run_logistic_regression_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 6: RUN NEURAL NETWORK MODEL
print("\nNEURAL NETWORKS TESTS")
run_neural_networks_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 7: SAVE OUTPUT
sys.stdout = sys.__stdout__
with open(f"Test_Results/{test_name}.txt", "w") as f:
    f.write(buffer.getvalue())

In [ ]:
# TEST CASE 4: FEMALE PARTICIPANTS, TARGET LABEL: AROUSAL
test_name = "RECOLA_Test_Case_4"

# STEP 1:  SETUP UP FILE SAVING FOR RESULTS
buffer = io.StringIO()
sys.stdout = buffer
print("-------- TEST CASE 4: FEMALE PARTICIPANTS, TARGET LABEL: AROUSAL --------\n")

# STEP 2: OBTAIN CASE DATASET
dataset_manager.temproary_recola = dataset_manager.split_by_gender(gender="Female")
dataset_manager.temproary_recola = dataset_manager.remove_class_label(label_to_keep="Arousal")
dataset = dataset_manager.save_custom_dataframe(file_name=f"{test_name}.csv")

# STEP 3: DATA PREP FOR MODELS
PARTICIPANT_GROUPS = list(dataset["Participant_Number"])
FEATURES = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
if "Class_Label_Valence" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Valence"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Valence"]
elif "Class_Label_Arousal" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Arousal"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Arousal"]
FOLDS = len(set(PARTICIPANT_GROUPS))

# STEP 4: DECIDE ON OUTPUT OF THE EXPERIMENTS
PRINT_INV_FEATURES  = False # True to display them, false not to

# STEP 5: RUN LOGISITIC REGRESSION MODEL
print("\nLOGISTIC REGRESSION TESTS")
run_logistic_regression_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 6: RUN NEURAL NETWORK MODEL
print("\nNEURAL NETWORKS TESTS")
run_neural_networks_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 7: SAVE OUTPUT
sys.stdout = sys.__stdout__
with open(f"Test_Results/{test_name}.txt", "w") as f:
    f.write(buffer.getvalue())

In [ ]:
# TEST CASE 5: FEMALE PARTICIPANTS, TARGET LABEL: VALENCE
test_name = "RECOLA_Test_Case_5"

# STEP 1:  SETUP UP FILE SAVING FOR RESULTS
buffer = io.StringIO()
sys.stdout = buffer
print("-------- TEST CASE 5: FEMALE PARTICIPANTS, TARGET LABEL: VALENCE --------\n")

# STEP 2: OBTAIN CASE DATASET
dataset_manager.temproary_recola = dataset_manager.split_by_gender(gender="Female")
dataset_manager.temproary_recola = dataset_manager.remove_class_label(label_to_keep="Valence")
dataset = dataset_manager.save_custom_dataframe(file_name=f"{test_name}.csv")

# STEP 3: DATA PREP FOR MODELS
PARTICIPANT_GROUPS = list(dataset["Participant_Number"])
FEATURES = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
if "Class_Label_Valence" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Valence"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Valence"]
elif "Class_Label_Arousal" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Arousal"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Arousal"]
FOLDS = len(set(PARTICIPANT_GROUPS))

# STEP 4: DECIDE ON OUTPUT OF THE EXPERIMENTS
PRINT_INV_FEATURES  = False # True to display them, false not to

# STEP 5: RUN LOGISITIC REGRESSION MODEL
print("\nLOGISTIC REGRESSION TESTS")
run_logistic_regression_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 6: RUN NEURAL NETWORK MODEL
print("\nNEURAL NETWORKS TESTS")
run_neural_networks_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 7: SAVE OUTPUT
sys.stdout = sys.__stdout__
with open(f"Test_Results/{test_name}.txt", "w") as f:
    f.write(buffer.getvalue())

In [ ]:
#  TEST CASE 6: MALE AND FEMALE ENVIRONS (2), TARGET LABEL: AROUSAL
test_name = "RECOLA_Test_Case_6"

# STEP 1:  SETUP UP FILE SAVING FOR RESULTS
buffer = io.StringIO()
sys.stdout = buffer
print("-------- TEST CASE 6: MALE AND FEMALE ENVIRONS (2), TARGET LABEL: AROUSAL --------\n")

# STEP 2: OBTAIN CASE DATASET
male_dataset = dataset_manager.split_by_gender(gender="Male")
female_dataset = dataset_manager.split_by_gender(gender="Female")
dataset_manager.temproary_recola = dataset_manager.simplify_environments(datasets=[male_dataset, female_dataset])
dataset_manager.temproary_recola = dataset_manager.remove_class_label(label_to_keep="Arousal")
dataset = dataset_manager.save_custom_dataframe(file_name=f"{test_name}.csv")

# STEP 3: DATA PREP FOR MODELS
PARTICIPANT_GROUPS = list(dataset["Participant_Number"])
FEATURES = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
if "Class_Label_Valence" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Valence"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Valence"]
elif "Class_Label_Arousal" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Arousal"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Arousal"]
FOLDS = len(set(PARTICIPANT_GROUPS))

# STEP 4: DECIDE ON OUTPUT OF THE EXPERIMENTS
PRINT_INV_FEATURES  = False # True to display them, false not to

# STEP 5: RUN LOGISITIC REGRESSION MODEL
print("\nLOGISTIC REGRESSION TESTS")
run_logistic_regression_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 6: RUN NEURAL NETWORK MODEL
print("\nNEURAL NETWORKS TESTS")
run_neural_networks_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 7: SAVE OUTPUT
sys.stdout = sys.__stdout__
with open(f"Test_Results/{test_name}.txt", "w") as f:
    f.write(buffer.getvalue())

In [ ]:
#  TEST CASE 7: MALE AND FEMALE ENVIRONS (2), TARGET LABEL: VALENCE
test_name = "RECOLA_Test_Case_7"

# STEP 1:  SETUP UP FILE SAVING FOR RESULTS
buffer = io.StringIO()
sys.stdout = buffer
print("-------- TEST CASE 7: MALE AND FEMALE ENVIRONS (2), TARGET LABEL: VALENCE --------\n")

# STEP 2: OBTAIN CASE DATASET
male_dataset = dataset_manager.split_by_gender(gender="Male")
female_dataset = dataset_manager.split_by_gender(gender="Female")
dataset_manager.temproary_recola = dataset_manager.simplify_environments(datasets=[male_dataset, female_dataset])
dataset_manager.temproary_recola = dataset_manager.remove_class_label(label_to_keep="Valence")
dataset = dataset_manager.save_custom_dataframe(file_name=f"{test_name}.csv")

# STEP 3: DATA PREP FOR MODELS
PARTICIPANT_GROUPS = list(dataset["Participant_Number"])
FEATURES = dataset.filter(regex=f'^{"ComPar"}|{"audio_speech"}|{"VIDEO"}|{"Face_detection"}|{"ECG"}|{"EDA"}', axis=1) 
if "Class_Label_Valence" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Valence"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Valence"]
elif "Class_Label_Arousal" in dataset.columns: 
    TARGET_LABEL = dataset["Class_Label_Arousal"]
    CONTINUOUS_TARGET_LABEL = dataset["Annotator_Arousal"]
FOLDS = len(set(PARTICIPANT_GROUPS))

# STEP 4: DECIDE ON OUTPUT OF THE EXPERIMENTS
PRINT_INV_FEATURES  = False # True to display them, false not to

# STEP 5: RUN LOGISITIC REGRESSION MODEL
print("\nLOGISTIC REGRESSION TESTS")
run_logistic_regression_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 6: RUN NEURAL NETWORK MODEL
print("\nNEURAL NETWORKS TESTS")
run_neural_networks_tests(dataset=dataset, groups=PARTICIPANT_GROUPS, features=FEATURES, target=TARGET_LABEL, continuous_target=CONTINUOUS_TARGET_LABEL, folds=FOLDS, display_inv_flag=PRINT_INV_FEATURES)

# STEP 7: SAVE OUTPUT
sys.stdout = sys.__stdout__
with open(f"Test_Results/{test_name}.txt", "w") as f:
    f.write(buffer.getvalue())

### AGAIN EXPERIMENT

In [ ]:
# SPLIT PER GAME 3 ENVIRONMENTS
# 30 participants whjo have played the the games (6 folds)
# if not work try with gorups of 5